# 09 · Dashboard 演示 V5（Streamlit）

## 这个 notebook 在做什么？

前面已经完成了模型从 V1 到 V4 的完整迭代：

| 版本 | 内容 | 产出 |
|------|------|------|
| V1 | 无泄漏赛前基线模型（战队历史战绩） | `output/models/v1_baseline.joblib` |
| V2 | BP 特征增强模型（英雄池 + 阵容） | `output/models/v2_bp_features.joblib` |
| V3 | 实时快照采集（模拟赛中数据） | `data/realtime/simulated_snapshots.csv` |
| V4 | 实时胜率时间切片模型 | `output/models/v4_realtime_model.joblib` |

现在，V5 的目标是：**把项目做成一个能演示的产品形态 Dashboard**。

## 为什么必须做 Dashboard？

面试官看到你简历写了「KPL 胜率预测」，心里会问：

> 「这个模型跑出来是个 notebook 表格，还是真的能给人用？」

Dashboard 能证明你具备：

1. **产品思维** — 你知道预测结果应该怎么呈现给用户
2. **工程能力** — 你能把模型从 notebook 抽出来做成可交互应用
3. **数据可视化** — 你会用 Plotly 做出专业的交互图表
4. **系统设计** — 多模型版本、多页面、数据缓存……体现架构意识

## 本 notebook 的产出

- `app/streamlit_app.py` — 完整可运行的 Dashboard 代码
- 4 个页面：首页概览 / 赛前预测 / 实时胜率曲线 / 模型对比

## 注意

这个 notebook **不会直接运行 Streamlit**（Streamlit 需要独立进程），而是：
1. 设计页面功能
2. 编写完整代码并写入文件
3. 提供运行说明

---
## 步骤 1 · Dashboard 功能规划

### 设计原则

好的数据产品 Dashboard 应该遵循这几个原则：

| 原则 | 说明 | 在我们项目中的体现 |
|------|------|-------------------|
| 信息层次 | 概览 → 详情 → 深入 | 首页看大数 → 选战队预测 → 看时序曲线 |
| 交互最小化 | 别让用户填太多东西 | 侧边栏选择，主区域展示结果 |
| 对比可见 | 关键数据放在一起比 | 双方战队指标并列、模型横向对比 |
| 容错友好 | 数据没准备好不能崩 | 每个模型/数据都有 fallback 提示 |

### 页面结构

```
┌─────────────────────────────────────────────────┐
│  侧边栏                                         │
│  ├─ 页面导航（Radio）                            │
│  ├─ 战队选择（赛前预测页）                        │
│  └─ battle_id 选择（实时曲线页）                  │
├─────────────────────────────────────────────────┤
│  📊 首页概览                                     │
│  ├─ 4 个指标卡：场次 / 战队 / 时间范围 / 快照数   │
│  ├─ 模型状态（3 个模型是否就绪）                  │
│  └─ 战队列表 + 胜率                              │
├─────────────────────────────────────────────────┤
│  🔮 赛前预测                                     │
│  ├─ 双方战队历史指标对比                          │
│  ├─ V1 模型预测结果 + 柱状图                     │
│  └─ V2 模型预测结果 + 柱状图                     │
├─────────────────────────────────────────────────┤
│  📈 实时胜率曲线                                  │
│  ├─ 选择 battle_id                               │
│  ├─ Plotly 折线图（双方胜率随时间变化）            │
│  └─ 实际胜负结果                                 │
├─────────────────────────────────────────────────┤
│  ⚖️ 模型对比                                     │
│  ├─ V1 / V2 / V4 指标表格                       │
│  ├─ 迭代思路说明                                 │
│  └─ 模型适用时间线                               │
└─────────────────────────────────────────────────┘
```

### 技术选型

| 组件 | 选择 | 理由 |
|------|------|------|
| 框架 | Streamlit | 快速出原型，Python 友好，面试能 5 分钟跑起来 |
| 图表 | Plotly | 交互式、美观、支持 hover |
| 缓存 | @st.cache_data | 数据只加载一次，切换页面不重复读 |
| 模型加载 | joblib | sklearn 生态标准序列化方式 |

---
## 步骤 2 · 编写完整 Dashboard 代码

下面这个 code cell 会将完整的 `streamlit_app.py` 写入 `app/` 目录。

### 代码结构说明

```
streamlit_app.py
├── 路径配置（PROJECT_ROOT, MODEL_DIR, ...）
├── 页面配置（st.set_page_config）
├── 数据加载函数（@st.cache_data）
│   ├── load_battles()
│   ├── load_matches()
│   ├── load_bp()
│   ├── load_snapshots()
│   └── load_model()
├── 特征工程函数
│   ├── build_team_history()  → 战队历史统计
│   ├── build_v1_features()  → V1 模型输入
│   └── build_v2_features()  → V2 模型输入
├── 侧边栏导航
└── 4 个页面（if/elif 分支）
    ├── 📊 首页概览
    ├── 🔮 赛前预测
    ├── 📈 实时胜率曲线
    └── ⚖️ 模型对比
```

运行下面的 cell，代码会自动写入 `app/streamlit_app.py`：

In [ ]:
import os
from pathlib import Path

# 确定项目根目录
PROJECT_ROOT = Path(os.getcwd()).parent
APP_DIR = PROJECT_ROOT / "app"
APP_DIR.mkdir(exist_ok=True)

# ════════════════════════════════════════════════════════════════════════════════
# 09 Dashboard 整体设计思路
# ════════════════════════════════════════════════════════════════════════════════
#
# 💡 核心问题："为什么要做 Dashboard？不做行不行？"
#
# 可以不做——但你的项目就只是一个 notebook，面试官看不到"产品形态"。
# Dashboard 存在的意义是回答这个问题：
#   "如果这个模型真的上线，用户怎么跟它交互？"
#
# ════════════════════════════════════════════════════════════════════════════════
# 产品设计思维：4 个页面对应 4 种用户需求
# ════════════════════════════════════════════════════════════════════════════════
#
# 页面 1 - 首页概览："数据准备好了吗？模型就绪了吗？"
#   → 给"管理者视角"看的——数据规模、模型状态、战队列表
#
# 页面 2 - 赛前预测："今天 A 队打 B 队，谁更可能赢？"
#   → 给"解说/观众"在比赛开始前看的
#
# 页面 3 - 实时胜率曲线："比赛打到一半，现在哪边更有优势？"
#   → 给"比赛进行中的观众"看的——核心页面
#
# 页面 4 - 模型对比："你的模型到底好不好？怎么改进的？"
#   → 给"面试官/技术评审"看的——证明你做了对比实验
#
# ════════════════════════════════════════════════════════════════════════════════
# 工程设计思维：这段代码最终要拼成一个 streamlit_app.py 写入磁盘
# ════════════════════════════════════════════════════════════════════════════════
#
# 代码结构：
#   section_1：导包 + 路径 + 页面配置
#   section_2：数据加载（带缓存，容错）
#   section_3：特征工程函数（从 notebook 复用到 Dashboard）
#   section_4：侧边栏导航
#   section_5~8：4 个页面的具体逻辑
#
# 每个 section 写成一个多行字符串，最后拼接写入文件。
#
# ---- 关键设计决策 ----
#
# 1. 为什么用 @st.cache_data？
#    → 数据只读一次，切换页面不重复加载（用户体验好）
#
# 2. 为什么每个 load 函数都检查文件存在？
#    → graceful degradation：即使某个模型没训练完，Dashboard 不会崩
#
# 3. 为什么特征工程要重新写函数而不是 import notebook？
#    → Dashboard 是独立运行的 .py 文件，不能依赖 notebook 状态
#
# 4. 为什么用 Plotly 而不是 Matplotlib？
#    → Streamlit 原生支持 Plotly 交互（hover、缩放），matplotlib 是静态图
#
# ════════════════════════════════════════════════════════════════════════════════
# 下面逐段编写，每段想清楚"这段在解决什么问题"再动手
# ════════════════════════════════════════════════════════════════════════════════


# ---- Section 1：导包 + 路径 + 页面配置 ----
# 思考：Streamlit 必须在最开头调 set_page_config，否则报错
# 路径用 Path(__file__).resolve().parents[1] 确保从 app/ 目录正确找到项目根
#
# 代码提醒：
#   from pathlib import Path
#   import numpy as np, pandas as pd
#   import plotly.express as px, plotly.graph_objects as go
#   import streamlit as st, joblib
#
#   PROJECT_ROOT = Path(__file__).resolve().parents[1]
#   PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
#   REALTIME_DIR  = PROJECT_ROOT / "data" / "realtime"
#   MODEL_DIR     = PROJECT_ROOT / "output" / "models"
#
#   st.set_page_config(page_title="KPL 实时胜率预测系统",
#                      layout="wide", initial_sidebar_state="expanded")
section_1_header = """
"""


# ---- Section 2：数据加载函数 ----
# 思考：5 个加载函数，全部容错（文件不存在返回 None/空 DataFrame）
# @st.cache_data 缓存 DataFrame，@st.cache_resource 缓存模型对象
# 区别：cache_data 序列化后缓存，cache_resource 直接引用（适合模型这种大对象）
#
# 代码提醒：
#   @st.cache_data
#   def load_battles():
#       path = PROCESSED_DIR / "battles.csv"
#       if not path.exists(): return pd.DataFrame()
#       return pd.read_csv(path)
#
#   @st.cache_resource
#   def load_model(name: str):
#       path = MODEL_DIR / name
#       if not path.exists(): return None
#       return joblib.load(path)
#
#   # load_matches, load_bp, load_snapshots 同理
section_2_data_loading = """
"""


# ---- Section 3：特征工程辅助函数 ----
# 思考：Dashboard 里需要实时构造特征给模型预测
# 需要 3 个函数：
#   build_team_history → 聚合战队历史数据（胜率、场均击杀等）
#   build_v1_features → 构造 V1 模型的输入（两队历史差值）
#   build_v2_features → 在 V1 基础上追加英雄池相关特征
# 关键：这些函数的输出必须和训练时的特征列完全对齐！
#
# 代码提醒：
#   def build_team_history(battles):
#       # 把 camp1 和 camp2 拆成两个子表，统一列名后 concat
#       c1 = battles[['camp1_team_name','camp1_kill_num','camp1_gold',...]].copy()
#       c1.columns = ['team_name', 'kills', 'gold', ...]
#       c1['win'] = (battles['win_camp'] == 1).astype(int)
#       # c2 同理...
#       long = pd.concat([c1, c2])
#       stats = long.groupby('team_name').agg(...)
#       stats['win_rate'] = stats['wins'] / stats['total']
#       return stats
#
#   def build_v1_features(team1_stats, team2_stats):
#       # 构造差值特征，返回单行 DataFrame
#       return pd.DataFrame([{
#           'win_rate_diff': team1_stats['win_rate'] - team2_stats['win_rate'],
#           ...
#       }])
section_3_feature_eng = """
"""


# ---- Section 4：侧边栏 + 数据加载 ----
# 思考：侧边栏是全局导航，radio 让用户选页面
# 数据在这里一次性加载（因为有缓存，不会重复读）
# 如果核心数据（battles）加载失败，直接 st.stop() 而不是让页面报错
#
# 代码提醒：
#   st.sidebar.title("🎮 KPL 胜率预测")
#   page = st.sidebar.radio("选择页面",
#       ["📊 首页概览", "🔮 赛前预测", "📈 实时胜率曲线", "⚖️ 模型对比"])
#   battles = load_battles()
#   if battles.empty:
#       st.error("未找到 battles.csv"); st.stop()
section_4_sidebar = """
"""


# ---- Section 5：首页概览 ----
# 思考：用户第一眼看到什么？
# 4 个 st.metric 卡片 → 一眼看清数据规模
# 模型状态 → 引导用户知道能做什么
# 战队一览表格 → 产品感
#
# 代码提醒：
#   if page == "📊 首页概览":
#       st.title("📊 项目概览")
#       c1, c2, c3, c4 = st.columns(4)
#       c1.metric("总比赛", f"{len(battles)} 场")
#       c2.metric("战队数", f"{n_teams} 支")
#       # ...
#       # 模型状态：
#       for name in ["v1_baseline.joblib", "v2_bp_features.joblib", "v4_realtime_model.joblib"]:
#           exists = (MODEL_DIR / name).exists()
#           col.metric(name.split('.')[0], "✅" if exists else "⏳")
section_5_home = """
"""


# ---- Section 6：赛前预测 ----
# 思考：用户选两支战队 → 展示历史对比 → 模型给出胜率预测
# 交互设计：两个 selectbox 并列，选完立刻出结果
# 容错：模型不存在就 st.info 提示，不崩溃
#
# 代码提醒：
#   elif page == "🔮 赛前预测":
#       team_stats = build_team_history(battles)
#       teams = team_stats.index.tolist()
#       col1, col2 = st.columns(2)
#       team1 = col1.selectbox("蓝方 (Camp1)", teams)
#       team2 = col2.selectbox("红方 (Camp2)", teams, index=1)
#       if team1 == team2: st.warning("请选不同战队"); st.stop()
#
#       # V1 预测：
#       model_v1 = load_model("v1_baseline.joblib")
#       if model_v1:
#           X = build_v1_features(team_stats.loc[team1], team_stats.loc[team2])
#           prob = model_v1.predict_proba(X)[:, 1][0]
#           fig = go.Figure(go.Bar(x=[team1, team2], y=[prob, 1-prob],
#                                  marker_color=["#1f77b4", "#d62728"]))
#           st.plotly_chart(fig)
section_6_predict = """
"""


# ---- Section 7：实时胜率曲线（最重要的页面）----
# 思考：这是面试时展示的核心页面
# 用户选一场比赛 → 看到胜率随时间变化的曲线
# 曲线在 0.5 以上 = camp1 优势，以下 = camp2 优势
#
# 代码提醒：
#   elif page == "📈 实时胜率曲线":
#       if snapshots.empty: st.warning("无快照数据"); st.stop()
#       model_v4 = load_model("v4_realtime_model.joblib")
#       bid = st.sidebar.selectbox("选择比赛", snapshots['battle_id'].unique())
#       game = snapshots[snapshots['battle_id'] == bid].sort_values('minute_bin')
#
#       if model_v4:
#           feat_cols = model_v4['feature_columns']
#           X = game[feat_cols].values
#           if model_v4['scaler']:
#               X = model_v4['scaler'].transform(X)
#           game['prob'] = model_v4['model'].predict_proba(X)[:, 1]
#
#           fig = go.Figure()
#           fig.add_trace(go.Scatter(x=game['minute_bin'], y=game['prob'],
#                                    mode='lines+markers', name='Camp1 胜率'))
#           fig.add_hline(y=0.5, line_dash="dash", line_color="gray")
#           st.plotly_chart(fig, use_container_width=True)
section_7_realtime = """
"""


# ---- Section 8：模型对比 ----
# 思考：面试官最想看的页面——你的模型迭代逻辑
# 横向对比 V1/V2/V4 的指标 → 证明"每次迭代都有改进"
# 时间线图 → 展示"不同模型适用于比赛的不同阶段"
#
# 代码提醒：
#   elif page == "⚖️ 模型对比":
#       st.title("⚖️ 模型迭代对比")
#       compare_data = [
#           {"版本": "V1", "特征": "战队历史", "场景": "赛前", "AUC": "..."},
#           {"版本": "V2", "特征": "+BP英雄", "场景": "BP后", "AUC": "..."},
#           {"版本": "V4", "特征": "+赛中局势", "场景": "比赛中", "AUC": "..."},
#       ]
#       st.dataframe(pd.DataFrame(compare_data))
#
#       # 时间线图：5 个节点标注模型适用时机
#       fig = go.Figure(go.Scatter(
#           x=[1, 2, 3, 4, 5], y=[0]*5, mode="markers+text",
#           text=["赛程公布", "BP结束", "开赛", "中期", "结束"],
#           textposition="top center", marker=dict(size=15)))
#       st.plotly_chart(fig)
#
#   st.sidebar.markdown("---")
#   st.sidebar.caption("KPL 实时胜率预测系统 · 2026")
section_8_compare = """
"""


# ============================================================
# 拼接完整代码并写入文件
# ============================================================
# 当你把上面的 TODO 9.1 ~ 9.8 都写完后，
# 把各段拼接成完整的 app_code

app_code = (
    section_1_header
    + section_2_data_loading
    + section_3_feature_eng
    + section_4_sidebar
    + section_5_home
    + section_6_predict
    + section_7_realtime
    + section_8_compare
)

app_path = APP_DIR / "streamlit_app.py"
app_path.write_text(app_code, encoding="utf-8")
print(f"✅ Dashboard 代码已写入: {app_path}")
print(f"   文件大小: {app_path.stat().st_size / 1024:.1f} KB")
print(f"   代码行数: {len(app_code.splitlines())}")

---
## 步骤 3 · 测试运行说明

### 环境依赖

确保已安装以下包：

```bash
pip install streamlit pandas plotly joblib scikit-learn
```

### 启动 Dashboard

在**项目根目录**下打开终端，运行：

```bash
streamlit run app/streamlit_app.py
```

Streamlit 会自动打开浏览器，默认地址是 `http://localhost:8501`。

### 运行前检查清单

| 检查项 | 说明 | 状态 |
|--------|------|------|
| `data/processed/battles.csv` | 必需，首页和赛前预测需要 | 运行 notebook 03/04 生成 |
| `data/processed/matches.csv` | 可选，首页展示时间范围 | 运行 notebook 03/04 生成 |
| `data/processed/bp.csv` | V2 预测需要 | 运行 notebook 03/04 生成 |
| `output/models/v1_baseline.joblib` | V1 赛前预测 | 运行 notebook 05 生成 |
| `output/models/v2_bp_features.joblib` | V2 赛前预测 | 运行 notebook 06 生成 |
| `data/realtime/simulated_snapshots.csv` | 实时曲线页面 | 运行 notebook 07 生成 |
| `output/models/v4_realtime_model.joblib` | V4 实时预测 | 运行 notebook 08 生成 |

### 常见问题

**Q: 页面显示「模型文件未找到」怎么办？**

A: 说明对应的训练 notebook 还没跑完。Dashboard 设计了 graceful fallback，不会崩溃，只是显示提示信息。按顺序跑完 05-08 notebook 即可。

**Q: 预测失败提示「特征列不匹配」？**

A: 说明你训练模型时用的特征列名和 Dashboard 里构造的不完全一致。解决方法：
1. 在训练 notebook 里 `print(model.feature_names_in_)` 看模型期望什么特征
2. 修改 Dashboard 中对应的 `build_vX_features()` 函数使其匹配

**Q: 能不能加新页面？**

A: 当然可以。在侧边栏的 `st.sidebar.radio` 里加一个选项，然后在下面加一个 `elif` 分支即可。推荐可以加的页面：
- 英雄 Ban/Pick 分析
- 战队历史对战记录
- 选手个人表现

In [ ]:
# 验证文件是否正确写入
import os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent
app_path = PROJECT_ROOT / "app" / "streamlit_app.py"

print("=" * 50)
print("文件验证")
print("=" * 50)
print(f"文件路径: {app_path}")
print(f"文件存在: {app_path.exists()}")
if app_path.exists():
    content = app_path.read_text(encoding="utf-8")
    print(f"文件大小: {len(content)} 字符")
    print(f"代码行数: {len(content.splitlines())} 行")
    print()
    print("前 10 行预览:")
    for line in content.splitlines()[:10]:
        print(f"  {line}")

---
## 步骤 4 · 总结报告 + ✅ 完成自检

### Dashboard 功能总结

| 页面 | 功能 | 核心组件 |
|------|------|----------|
| 📊 首页概览 | 项目统计、模型状态、战队列表 | `st.metric` + `st.dataframe` |
| 🔮 赛前预测 | 选战队 → V1/V2 预测胜率 | `st.selectbox` + `plotly.Bar` |
| 📈 实时胜率曲线 | 选比赛 → V4 胜率时序图 | `st.selectbox` + `plotly.Scatter` |
| ⚖️ 模型对比 | V1/V2/V4 指标横向表格 | `st.dataframe` + 时间线图 |

### 技术亮点（面试可讲）

1. **多阶段模型产品化**：从赛前到实时的 3 个模型统一到一个 Dashboard
2. **数据缓存**：`@st.cache_data` 避免重复加载，提升交互速度
3. **Graceful Degradation**：数据/模型缺失时不崩溃，给出明确提示
4. **交互式图表**：Plotly 支持 hover、缩放，用户体验好
5. **特征工程复用**：Dashboard 的特征构造函数与训练保持一致

### 简历表达建议

> 基于 Streamlit 搭建交互式 Dashboard，集成 V1（赛前基线）、V2（BP增强）、V4（实时快照）三阶段模型，
> 支持战队对阵预测和实时胜率曲线展示，使用 Plotly 实现交互可视化，
> 通过 @st.cache_data 优化数据加载性能，具备容错降级能力。

### ✅ 完成自检

- [x] Dashboard 代码完整写入 `app/streamlit_app.py`
- [x] 4 个页面功能齐全
- [x] 使用 `@st.cache_data` 缓存数据加载
- [x] 使用 `joblib.load` 加载模型
- [x] 使用 Plotly 交互式图表
- [x] 所有文本中文
- [x] 模型/数据缺失时有 fallback 提示
- [x] 代码结构清晰，初级开发者可读

### 下一步

1. 按顺序跑完 notebook 05-08，生成所有模型文件
2. 终端运行 `streamlit run app/streamlit_app.py`
3. 截图 / 录屏作为简历附件
4. （进阶）部署到 Streamlit Cloud，简历附在线链接